In [1]:
try:
    import nba_api
    print(f"[✓] nba_api version: {nba_api.__version__}")
except:
    print("[!] Không xác định được version nba_api")

[✓] nba_api version: 1.11.4


In [2]:
import pandas as pd
import time
import os
from datetime import datetime, timedelta
import inspect

from nba_api.stats.endpoints import (
    LeagueGameFinder, 
    BoxScoreAdvancedV2,
    LeagueStandingsV3
)
from nba_api.stats.static import teams

In [3]:
SEASONS = ["2021-22", "2022-23", "2023-24", "2024-25", "2025-26"]
SEASON_TYPE = 'Regular Season'
REQUEST_DELAY = 0.8

OUTPUT_DIR = 'nba_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Help function

In [4]:
def safe_request(func, max_retries=3, **kwargs):
    """
    Gọi API với retry và delay.
    Tự động loại bỏ các tham số không hợp lệ cho endpoint.
    """
    # Lấy danh sách tham số mà endpoint chấp nhận
    valid_params = set(inspect.signature(func.__init__).parameters.keys())
    valid_params.discard("self")

    # Lọc chỉ giữ các kwargs hợp lệ
    filtered_kwargs = {}
    for key, value in kwargs.items():
        if key in valid_params:
            filtered_kwargs[key] = value
        else:
            print(f"  [!] Bỏ qua tham số không hợp lệ: '{key}'")
            similar = [p for p in valid_params if key[:5] in p]
            if similar:
                print(f"      Có phải bạn muốn dùng: {similar}?")

    for attempt in range(max_retries):
        try:
            time.sleep(REQUEST_DELAY)
            result = func(**filtered_kwargs)
            return result
        except Exception as e:
            print(f"  [!] Lỗi (lần {attempt + 1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                wait_time = REQUEST_DELAY * (attempt + 2)
                print(f"  [*] Chờ {wait_time:.1f}s rồi thử lại...")
                time.sleep(wait_time)
            else:
                print(f"  [X] Bỏ qua sau {max_retries} lần thử.")
                return None

In [5]:
def get_all_teams():
    # Lấy danh sách 30 đội NBA
    nba_teams = teams.get_teams()
    df = pd.DataFrame(nba_teams)
    print(f'Có {len(df)} đội NBA')
    return df

# 2. NHÓM 1 + 2: THÔNG TIN TRẬN ĐẤU + THỐNG KÊ CƠ BẢN

In [6]:
def crawl_game_info_and_basic_stats(seasons=SEASONS):
    '''
    Thu thập Nhóm 1 (Game info) và Nhóm 2 (Traditional Stats) cùng lúc

    Endpoint: LeagueGameFinder
    Trả về: DataFrame chứa mỗi dòng = 1 đội trong 1 trận đấu
    -> Mỗi trận đấu sẽ có 2 dòng (đội nhà + đội khách)

    Các cột quan trọng:
    - GAME_ID: mã trận đấu (dùng để join với các bảng khác)
    - GAME_DATE: ngày thi đấu
    - TEAM_ID, TEAM_NAME: thông tin đội
    - MATCHUP: "GSW vs. LAL" (nhà) hoặc "GSW @ LAL" (khách)
    - WL: kết quả "W" hoặc "L" → đây chính là LABEL
    - PTS, REB, AST, STL, BLK, TOV, ... : thống kê cơ bản
    - FGM, FGA, FG_PCT, FG3M, FG3A, FG3_PCT, ... : hiệu suất ném
    '''
    print("\n" + "=" * 60)
    print("CRAWL NHÓM 1 & 2: Game Info + Traditional Box Score")
    print("=" * 60)

    all_games = []

    for season in seasons:
        print(f'[->] Đang crawl mùa giải {season}...')

        result = safe_request(
            LeagueGameFinder,
            season_nullable=season,
            season_type_nullable=SEASON_TYPE,
            league_id_nullable='00',
        )

        if result is None:
            print(f"  [X] Không lấy được dữ liệu mùa {season}")
            continue

        df = result.get_data_frames()[0]
        df['SEASON'] = season
        all_games.append(df)
        print(f"  [✓] Lấy được {len(df)} dòng (mỗi trận có 2 dòng)")

    if not all_games:
        print("[X] Không có dữ liệu nào được crawl!")
        return pd.DataFrame()

    # Gộp tất cả mùa giải
    games_df = pd.concat(all_games, ignore_index=True)

    # Chuyển GAME_DATE sang datetime
    games_df['GAME_DATE'] = pd.to_datetime(games_df['GAME_DATE'])

    # Tạo cột IS_HOME: dựa vào MATCHUP
    # 'vs.' = sân nhà, '@' = sân khách
    games_df['IS_HOME'] = games_df['MATCHUP'].apply(
        lambda x: 1 if "vs." in x else 0
    )

    # Tạo cột WIN (label): W = 1, L = 0
    games_df['WIN'] = games_df['WL'].apply(lambda x: 1 if x == 'W' else 0)

    # Sắp xếp thời gian
    games_df = games_df.sort_values(['GAME_DATE', 'GAME_ID']).reset_index(drop=True)

    # Lưu file
    output_path = os.path.join(OUTPUT_DIR, "group1_2_games_basic_stats.csv")
    games_df.to_csv(output_path, index=False)

    print(f"\n[✓] NHÓM 1 & 2: Tổng cộng {len(games_df)} dòng")
    print(f"    Số trận đấu: {games_df['GAME_ID'].nunique()}")
    print(f"    Khoảng thời gian: {games_df['GAME_DATE'].min()} → {games_df['GAME_DATE'].max()}")
    print(f"    Đã lưu: {output_path}")

    return games_df

# 3. NHÓM 4: CONTEXTUAL FEATURES

In [7]:
def build_contextual_features(games_df):
    '''
    Tính toán nhóm 4: Contextual Features từ nhóm 1, 2

    Đây là bước FEATURE ENGINEERING đầu tiên
    Các feature được tạo:
    1. IS_HOME: đã có từ nhóm 1, 2
    2. REST_DAYS: số ngày nghỉ kể từ trận trước
    3. IS_BACK_TO_BACK: thi đấu 2 ngày liên tiếp (REST_DAYS = 0 hoặc 1)
    4. CURRENT_WIN_PCT: tỷ lệ thắng tính đến trước trận hiện tại
    5. CURRENT_WIN_STREAK: chuỗi thắng/thua hiện tại
    6. GAMES_PLAYED_SEASON: số trận đã đấu trong mùa (mệt mỏi tích lũy)
    '''
    print("\n" + "=" * 60)
    print("TÍNH NHÓM 4: Contextual Features")
    print("=" * 60)

    # Sắp xếp theo đội + mùa + thời gian
    df = games_df.copy()
    df = df.sort_values(['TEAM_ID', 'SEASON', 'GAME_DATE']).reset_index(drop=True)

    # 1. ----- REST DAYS ------
    # [FIX] Groupby cả SEASON để mỗi trận đầu mùa đều là NaN → fill 7
    # Trước đó chỉ groupby TEAM_ID → trận đầu mùa mới tính REST_DAYS
    # từ trận cuối mùa trước (150-200 ngày nghỉ hè) → sai
    df['PREV_GAME_DATE'] = df.groupby(['TEAM_ID', 'SEASON'])['GAME_DATE'].shift(1)
    df['REST_DAYS'] = (df['GAME_DATE'] - df['PREV_GAME_DATE']).dt.days

    # Trận đầu mỗi mùa không có trận trước -> gán nghỉ 7 ngày (giả định)
    df['REST_DAYS'] = df['REST_DAYS'].fillna(7)

    # 2. ------ IS_BACK_TO_BACK ------
    df['IS_B2B'] = (df['REST_DAYS'] <= 1).astype(int)

    # 3. ------- CURRENT_WIN_PCT -----
    # Dùng expanding mean rồi shift 1 để không bị data leakage
    df['CUMULATIVE_WINS'] = df.groupby(['TEAM_ID', 'SEASON'])['WIN'].cumsum()
    df['GAMES_PLAYED_SEASON'] = df.groupby(['TEAM_ID', 'SEASON']).cumcount() + 1

    # Shift (no leakage)
    df['WINS_BEFORE'] = df.groupby(['TEAM_ID', 'SEASON'])['CUMULATIVE_WINS'].shift(1)
    df['GP_BEFORE'] = df.groupby(['TEAM_ID', 'SEASON'])['GAMES_PLAYED_SEASON'].shift(1)
    df['CURRENT_WIN_PCT'] = df['WINS_BEFORE'] / df['GP_BEFORE']

    # Trận đầu tiên của mùa: chưa có lịch sử → gán 0.5
    df['CURRENT_WIN_PCT'] = df['CURRENT_WIN_PCT'].fillna(0.5)

    # 4. --------- WIN_STREAK ----------
    # Dương = đang thắng liên tiếp, Âm = đang thua liên tiếp
    def calc_streak(group):
        streaks = []
        current_streak = 0
        for _, row in group.iterrows():
            # Lưu streak trước khi cập nhật (no leakage)
            streaks.append(current_streak)
            if row['WIN'] == 1:
                current_streak = current_streak + 1 if current_streak > 0 else 1
            else:
                current_streak = current_streak - 1 if current_streak < 0 else -1
        return pd.Series(streaks, index=group.index)

    # [FIX] Thêm include_groups=False để tránh DeprecationWarning
    df['WIN_STREAK'] = df.groupby(
        ['TEAM_ID', 'SEASON'], group_keys=False
    ).apply(calc_streak, include_groups=False)

    df = df.drop(columns=[
        'PREV_GAME_DATE', 'CUMULATIVE_WINS', 'WINS_BEFORE', 'GP_BEFORE'
    ])

    # Lưu file
    output_path = os.path.join(OUTPUT_DIR, "group4_contextual_features.csv")
    context_cols = [
        "GAME_ID", "TEAM_ID", "TEAM_NAME", "GAME_DATE", "SEASON",
        "IS_HOME", "REST_DAYS", "IS_B2B", "CURRENT_WIN_PCT",
        "WIN_STREAK", "GAMES_PLAYED_SEASON"
    ]
    df[context_cols].to_csv(output_path, index=False)

    print(f"[✓] NHÓM 4: Đã tính contextual features")
    print(f"    Các feature mới: REST_DAYS, IS_B2B, CURRENT_WIN_PCT, WIN_STREAK, GAMES_PLAYED_SEASON")
    print(f"    Đã lưu: {output_path}")

    return df

# 4. GỘP TẤT CẢ 4 NHÓM - DATASET CUỐI CÙNG

In [8]:
def merge_all_groups(games_df, advanced_df):
    '''
    Merge 4 nhóm dữ liệu thành 1 dataset duy nhất

    Khóa join: GAME_ID + TEAM_ID
    -> Mỗi dòng = 1 đội trong 1 trận đấu

    Sau bước này, cần thêm 1 bước nữa để ghép 2 đội trong cùng 1 trận lại thành 1 dòng
    (ở bước Feature Engineering)
    '''
    print("\n" + "=" * 60)
    print("MERGE TẤT CẢ NHÓM → DATASET CUỐI CÙNG")
    print("=" * 60)

    # Contextual features
    context_df = build_contextual_features(games_df)

    # Merge advanced stats vào
    if not advanced_df.empty:
        adv_cols_to_keep = [
            'GAME_ID', 'TEAM_ID',
            'OFF_RATING', 'DEF_RATING', 'NET_RATING',
            'PACE', 'TS_PCT', 'EFG_PCT',
            'AST_RATIO', 'OREB_PCT', 'DREB_PCT',
            'REB_PCT', 'E_TM_TOV_PCT'
        ]

        adv_cols_available = [c for c in adv_cols_to_keep if c in advanced_df.columns]
        adv_subset = advanced_df[adv_cols_available]

        merged = context_df.merge(
            adv_subset,
            on=['GAME_ID', 'TEAM_ID'],
            how='left',
        )
        print(f"[✓] Đã merge advanced stats")
    else:
        merged = context_df
        print("[!] Bỏ qua advanced stats (không có dữ liệu)")

    # Lưu dataset cuối cùng
    output_path = os.path.join(OUTPUT_DIR, "nba_full_dataset.csv")
    merged.to_csv(output_path, index=False)

    print(f"\n[✓] DATASET CUỐI CÙNG:")
    print(f"    Shape: {merged.shape}")
    print(f"    Số trận: {merged['GAME_ID'].nunique()}")
    print(f"    Số đội: {merged['TEAM_ID'].nunique()}")
    print(f"    Khoảng thời gian: {merged['GAME_DATE'].min()} → {merged['GAME_DATE'].max()}")
    print(f"    Số cột: {merged.shape[1]}")
    print(f"    Đã lưu: {output_path}")
    print(f"\n    Các cột:\n    {list(merged.columns)}")

    return merged

# 6. RUN

In [9]:
if __name__ == "__main__":
  print("╔" + "═" * 58 + "╗")
  print("║   NBA DATA CRAWLER - Dự đoán kết quả trận đấu NBA      ║")
  print("╚" + "═" * 58 + "╝")
  print(f"\nCấu hình:")
  print(f"  - Mùa giải: {SEASONS}")
  print(f"  - Loại: {SEASON_TYPE}")
  print(f"  - Delay giữa request: {REQUEST_DELAY}s")
  print(f"  - Thư mục output: {OUTPUT_DIR}/")

  # Bước 1 + 2: Crawl Game Info + Basic Stats
  games_df = crawl_game_info_and_basic_stats()

  if games_df.empty:
      print("\n⚠️  Không có dữ liệu. Các bước sau sẽ bị bỏ qua.")


  # Bước 4: Merge tất cả
  final_df = merge_all_groups(games_df, pd.DataFrame())


  print("\n" + "=" * 60)
  print("HOÀN TẤT! Kiểm tra thư mục:", OUTPUT_DIR)
  print("=" * 60)
  print(f"""
Các file đã tạo:
  1. {OUTPUT_DIR}/group1_2_games_basic_stats.csv  → Nhóm 1 & 2
  3. {OUTPUT_DIR}/group4_contextual_features.csv   → Nhóm 4
  4. {OUTPUT_DIR}/nba_full_dataset.csv             → Dataset gộp
  5. {OUTPUT_DIR}/standings_YYYY-YY.csv            → Bảng xếp hạng
""")

╔══════════════════════════════════════════════════════════╗
║   NBA DATA CRAWLER - Dự đoán kết quả trận đấu NBA      ║
╚══════════════════════════════════════════════════════════╝

Cấu hình:
  - Mùa giải: ['2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
  - Loại: Regular Season
  - Delay giữa request: 0.8s
  - Thư mục output: nba_data/

CRAWL NHÓM 1 & 2: Game Info + Traditional Box Score
[->] Đang crawl mùa giải 2021-22...
  [✓] Lấy được 2460 dòng (mỗi trận có 2 dòng)
[->] Đang crawl mùa giải 2022-23...
  [✓] Lấy được 2460 dòng (mỗi trận có 2 dòng)
[->] Đang crawl mùa giải 2023-24...
  [✓] Lấy được 2460 dòng (mỗi trận có 2 dòng)
[->] Đang crawl mùa giải 2024-25...
  [✓] Lấy được 2460 dòng (mỗi trận có 2 dòng)
[->] Đang crawl mùa giải 2025-26...
  [✓] Lấy được 2460 dòng (mỗi trận có 2 dòng)

[✓] NHÓM 1 & 2: Tổng cộng 12300 dòng
    Số trận đấu: 6150
    Khoảng thời gian: 2021-10-19 00:00:00 → 2026-04-12 00:00:00
    Đã lưu: nba_data\group1_2_games_basic_stats.csv

MERGE TẤT CẢ N